In [ ]:
import os
import re
import pandas as pd
import rispy

# Set the root directory containing all the subfolders
root_dir = r'C:\Users\yourpathto\ArticlesData'   # <-- Replace this with your actual path


# Initialize a list to collect all entries
all_entries = []

# Regex pattern to extract Q and T from folder name
folder_pattern = re.compile(r'Q(\d+)xT(\d+)\s+(Scopus|DTU FindIt Eng|Google Scholar|DTU FindIt Esp)')
# Walk through each subfolder
for folder_name in os.listdir(root_dir):
    match = folder_pattern.match(folder_name)
    if not match:
        continue  # Skip folders not matching the expected pattern

    Q_val, T_val, database = match.groups()
    folder_path = os.path.join(root_dir, folder_name)

    # Find the .ris file in the folder
    ris_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.ris')]
    if not ris_files:
        continue  # Skip if no RIS file is found

    ris_path = os.path.join(folder_path, ris_files[0])
    
    # Parse RIS file
    with open(ris_path, 'r', encoding='utf-8') as ris_file:
        entries = rispy.load(ris_file)

        for entry in entries:
            # Add Q, T, and database to each entry
            entry['Q'] = int(Q_val)
            entry['T'] = int(T_val)
            entry['Database'] = database
            # Check for journal name using multiple possible tags
            journal = (
                entry.get('secondary_title') or  # T2 tag in RIS (standard for journal names)
                entry.get('journal_name') or     # Backup field
                entry.get('alternate_title3') or # Another possible field
                ''  # Default empty string if none found
            )
            #Save the abstract too in a new field
            entry['abstract_text'] = entry.get('abstract', '')

            # If a journal name is found, clean it up
            entry['journal_name'] = journal.strip() if journal else ''
            all_entries.append(entry)

# Convert to DataFrame
df = pd.DataFrame(all_entries)

# Rename or select useful columns if needed
columns_of_interest = {
    'title': 'Title',
    'authors': 'Authors',
    'year': 'Year',
    'journal_name': 'Journal',
    'doi': 'DOI',
    'url': 'URL',
    'publisher': 'Publisher',
    'type_of_reference': 'ReferenceType',
    'Database': 'Database',
    'Q': 'Q',
    'T': 'T',
    'abstract_text': 'Abstract'
}

# Filter and rename only those columns present in the data
final_columns = {k: v for k, v in columns_of_interest.items() if k in df.columns}
df = df[list(final_columns.keys())].rename(columns=final_columns)

def filter_by_year(df, min_year):
    # Ensure 'year' is numeric
    df['year'] = pd.to_numeric(df['Year'], errors='coerce')
    # Keep rows where year is NaN (missing) OR year >= min_year
    filtered_df = df[df['year'].isna() | (df['year'] >= min_year)].copy()
    return filtered_df.reset_index(drop=True)

df = filter_by_year(df, 2010)
# Show result
df
# Optional: Export to CSV
# df.to_csv("literature_review_data.csv", index=False)


,Title,Authors,Year,Journal,DOI,Publisher,ReferenceType,Database,Q,T,Abstract,year
0,Training quantum Boltzmann machines with the β...,"[Huijgen, Onno, Coopmans, Luuk, Najafi, Peyman...",2024,Machine Learning: Science and Technology,10.1088/2632-2153/ad370f,Institute of Physics,JOUR,DTU FindIt Eng,11,5,The quantum Boltzmann machine (QBM) is a gener...,2024.0
1,Andreev non-Hermitian Hamiltonian for open Jos...,"[Capecelatro, R., Marciani, M., Campagnano, G....",2025,Physical Review B,10.1103/PhysRevB.111.064517,American Physical Society,JOUR,DTU FindIt Eng,1,1,We investigate the transport properties of ope...,2025.0
2,Anti-scarring from eigenstate stacking in a ch...,"[Lu, Zhongling, Graf, Anton M., Heller, Eric J...",2025,,NaN,NaN,JOUR,DTU FindIt Eng,1,1,The early-time dynamics of many-body quantum c...,2025.0
3,Apparent teleportation of indistinguishable pa...,"[Gazdzicki, Marek, Kikola, Daniel, Pidhurskyi,...",2025,,NaN,NaN,JOUR,DTU FindIt Eng,1,1,"Teleportation, introduced in science fiction l...",2025.0
4,Beyond nearest-neighbor universality of spectr...,"[Kundu, Debojyoti, Kumar, Santosh, Sen Gupta, ...",2025,Chaos,10.1063/5.0234333,AIP Publishing LLC,JOUR,DTU FindIt Eng,1,1,Discerning chaos in quantum systems is an impo...,2025.0
...,...,...,...,...,...,...,...,...,...,...,...,...
22499,2013 2nd International Conference on Civil Eng...,NaN,2014,Advanced Materials Research,NaN,NaN,JOUR,Scopus,7,9,,2014.0
22500,"15th Metaheuristics International Conference, ...",NaN,2024,Lecture Notes in Computer Science (including s...,NaN,NaN,JOUR,Scopus,7,9,,2024.0
22501,"15th Metaheuristics International Conference, ...",NaN,2024,Lecture Notes in Computer Science (including s...,NaN,NaN,JOUR,Scopus,7,9,,2024.0
22502,3rd International Conference on Futuristic Tre...,NaN,2021,Communications in Computer and Information Sci...,NaN,NaN,JOUR,Scopus,7,9,,2021.0


In [2]:
database_counts = df['Database'].value_counts()

# Print the result
print(database_counts)
#print total number of entries
print(f'Total number of entries: {len(df)}')


Database
Scopus            14996
DTU FindIt Eng     7005
Google Scholar      469
DTU FindIt Esp       34
Name: count, dtype: int64
Total number of entries: 22504


In [3]:
def deduplicate_articles_with_database_priority(df):
    # Define database priority (lower number = higher priority)
    db_priority = {
        'Google Scholar': 1,
        'Scopus': 2,
        'DTU Find It Esp': 3,
        'DTU Find It Eng': 4
    }

    # Assign numeric Q and T for sorting
    df['db_priority'] = df['Database'].map(db_priority)

    # Create deduplication keys
    df['dedup_key'] = df['DOI'].fillna('').str.lower().str.strip()
    df['alt_key'] = df['Title'].str.lower().str.strip() + '_' + df['Year'].astype(str)

    # Sort by deduplication priority: Higher Q, then T, then better database
    df = df.sort_values(by=['dedup_key', 'alt_key', 'Q', 'T', 'db_priority'], ascending=[True, True, False, False, True])

    # Use groupby to keep the first occurrence of each duplicate key
    seen_keys = set()
    keep_rows = []

    for idx, row in df.iterrows():
        key = row['dedup_key'] if row['dedup_key'] else row['alt_key']
        if key not in seen_keys:
            seen_keys.add(key)
            keep_rows.append(idx)

    deduped_df = df.loc[keep_rows].drop(columns=['db_priority', 'dedup_key', 'alt_key']).reset_index(drop=True)
    return deduped_df

In [4]:
deduped_df = deduplicate_articles_with_database_priority(df)

In [5]:
def filter_by_year(df, min_year):
    # Ensure 'year' is numeric
    df['year'] = pd.to_numeric(df['Year'], errors='coerce')
    # Keep rows where year is NaN (missing) OR year >= min_year
    filtered_df = df[df['year'].isna() | (df['year'] >= min_year)].copy()
    return filtered_df.reset_index(drop=True)

filtered_df = filter_by_year(deduped_df, 2010)
filtered_df

,Title,Authors,Year,Journal,DOI,Publisher,ReferenceType,Database,Q,T,Abstract,year
0,$\mathrm{\Lambda_{c}^{+}}$ baryon production m...,"[Norman, Jaime]",2018,,NaN,NaN,JOUR,DTU FindIt Eng,1,1,"Quantum chromodynamics, the quantum field theo...",2018.0
1,10th EAI International Conference on Industria...,NaN,2024,Lecture Notes of the Institute for Computer Sc...,NaN,NaN,JOUR,Scopus,7,7,The proceedings contain 18 papers. The special...,2024.0
2,10th International Conference on Emerging Ubiq...,NaN,2019,Procedia Computer Science,NaN,NaN,CONF,Scopus,7,6,The proceedings contain 126 papers. The topics...,2019.0
3,10th International Conference on Monte Carlo a...,NaN,2013,Springer Proceedings in Mathematics and Statis...,NaN,NaN,CONF,Scopus,1,6,The proceedings contain 34 papers. The special...,2013.0
4,10th International Symposium on Intelligence C...,NaN,2019,Communications in Computer and Information Sci...,NaN,NaN,JOUR,Scopus,1,6,The proceedings contain 32 papers. The special...,2019.0
...,...,...,...,...,...,...,...,...,...,...,...,...
13463,Large-scale mesoscopic transport in nanostruct...,"[Hai-Jing, Zhang]",2013,Wuli,10.7693/wl201.30701,NaN,JOUR,DTU FindIt Eng,1,1,,2013.0
13464,Mechanistic pathways of mercury removal fromth...,"[Silva, P.J., Rodrigues, V.]",2015,PeerJ,10.7717/peerj.1127,NaN,JOUR,Scopus,1,5,Bacterial populations present in Hg-rich envir...,2015.0
13465,"Pharmacoscreening, molecular dynamics, and qua...","[Janakiraman, V., Sudhan, M., Wani, A., Ahmad,...",2023,PeerJ,10.7717/peerj.16481,NaN,JOUR,Scopus,1,6,"Background. Exosomes, microvesicles, carry and...",2023.0
13466,Sub-celluiar localization of PELPK1 in Arabido...,"[Rashid, A.]",2014,Molekuliarnaia biologiia,10.7868/s0026898414020165,NaN,JOUR,Scopus,1,1,"PELPK1, a novel Arabidopsis thaliana gene was ...",2014.0


In [6]:
import re
import pandas as pd

def find_paper_by_title_loose(df, title_query):
    """
    Loosely compare all titles to a query and rank by keyword overlap.
    Prints all entries, sorted by relevance.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame produced by your existing code
    title_query : str
        Full paper title or a descriptive fragment

    Returns
    -------
    pd.DataFrame
        All rows with a MatchScore column
    """
    if 'Title' not in df.columns:
        raise ValueError("Column 'Title' not found in DataFrame.")

    # Normalize function
    def normalize(text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return re.sub(r'\s+', ' ', text).strip()

    query_words = set(normalize(title_query).split())

    def match_score(title):
        if pd.isna(title):
            return 0
        title_words = set(normalize(title).split())
        return len(query_words & title_words)

    result = df.copy()
    result['MatchScore'] = result['Title'].apply(match_score)

    return result[
        ['MatchScore', 'Title', 'Q', 'T', 'Database', 'Year', 'Journal']
    ].sort_values('MatchScore', ascending=False)


In [7]:
def filter_out_by_keywords(df, keywords, title_col, verbose=True):
    """Filter out articles whose titles contain specific keywords.
    
    Args:
        df (DataFrame): Input DataFrame containing article information
        keywords (list): List of keywords to filter out
        title_col (str): Column name containing article titles
        verbose (bool): Whether to print statistics about filtering
        
    Returns:
        DataFrame: Filtered DataFrame with matching articles removed
    """
    if df.empty or not keywords:
        return df
    
    # Initial count
    initial_count = len(df)
    
    # Combine keywords into a single regex pattern (case-insensitive)
    pattern = re.compile('|'.join(re.escape(k) for k in keywords), flags=re.IGNORECASE)
    
    # Create a boolean mask for rows to keep (those NOT containing keywords)
    mask = ~df[title_col].fillna('').str.contains(pattern, na=False, regex=True)
    
    # Apply the mask
    filtered_df = df[mask].copy().reset_index(drop=True)
    
    # Report statistics if verbose
    if verbose:
        excluded_count = initial_count - len(filtered_df)
        print(f"Filtered out {excluded_count} articles ({excluded_count/initial_count*100:.1f}%) "
              f"containing these keywords in titles: {', '.join(keywords)}")
        
        # Find some examples of excluded articles for verification
        if excluded_count > 0:
            excluded_df = df[~mask]
            print("\nExamples of excluded articles:")
            #for _, row in excluded_df.head(90).iterrows():
            for i, (_, row) in enumerate(excluded_df.head(90).iterrows()):
                title = row[title_col]
                matching_keywords = [k for k in keywords if re.search(k, title, re.IGNORECASE)]
                print(f"- '{title}' (matched: {', '.join(matching_keywords)})")
    
    return filtered_df

In [8]:
keywords_to_exclude = ['chem', 'metal', 'molec', 'bio', 'cond', 'cataly', 'material', 'solid', 'spectro', 'fluor', 'pharma']
filtered_df = filter_out_by_keywords(filtered_df, keywords_to_exclude, 'Journal', verbose=True)
filtered_df

Filtered out 4275 articles (31.7%) containing these keywords in titles: chem, metal, molec, bio, cond, cataly, material, solid, spectro, fluor, pharma

Examples of excluded articles:
- 'Lecture Notes in Computer Science (including subseries Lecture Notes in Artificial Intelligence and Lecture Notes in Bioinformatics)' (matched: bio)
- 'Lecture Notes in Computer Science (including subseries Lecture Notes in Artificial Intelligence and Lecture Notes in Bioinformatics)' (matched: bio)
- 'Lecture Notes in Computer Science (including subseries Lecture Notes in Artificial Intelligence and Lecture Notes in Bioinformatics)' (matched: bio)
- 'Lecture Notes in Computer Science (including subseries Lecture Notes in Artificial Intelligence and Lecture Notes in Bioinformatics)' (matched: bio)
- 'International Conference on Simulation of Semiconductor Processes and Devices, SISPAD' (matched: cond)
- 'Lecture Notes in Computer Science (including subseries Lecture Notes in Artificial Intelligence and 

,Title,Authors,Year,Journal,DOI,Publisher,ReferenceType,Database,Q,T,Abstract,year
0,$\mathrm{\Lambda_{c}^{+}}$ baryon production m...,"[Norman, Jaime]",2018,,NaN,NaN,JOUR,DTU FindIt Eng,1,1,"Quantum chromodynamics, the quantum field theo...",2018.0
1,10th EAI International Conference on Industria...,NaN,2024,Lecture Notes of the Institute for Computer Sc...,NaN,NaN,JOUR,Scopus,7,7,The proceedings contain 18 papers. The special...,2024.0
2,10th International Conference on Emerging Ubiq...,NaN,2019,Procedia Computer Science,NaN,NaN,CONF,Scopus,7,6,The proceedings contain 126 papers. The topics...,2019.0
3,10th International Conference on Monte Carlo a...,NaN,2013,Springer Proceedings in Mathematics and Statis...,NaN,NaN,CONF,Scopus,1,6,The proceedings contain 34 papers. The special...,2013.0
4,10th International Symposium on Intelligence C...,NaN,2019,Communications in Computer and Information Sci...,NaN,NaN,JOUR,Scopus,1,6,The proceedings contain 32 papers. The special...,2019.0
...,...,...,...,...,...,...,...,...,...,...,...,...
9188,Calculation of Rate Coefficients for Unsymmetr...,"[Yin, G., You, J., Hu, E., Huang, Z.]",2024,Hsi-An Chiao Tung Ta Hsueh/Journal of Xi'an Ji...,10.7652/xjtuxb202412020,NaN,JOUR,Scopus,1,5,To address the issue of inaccuracies in the ki...,2024.0
9189,Simulation of vehicle distribution path optimi...,"[Ding, Y.]",2021,Shenyang Gongye Daxue Xuebao/Journal of Shenya...,10.7688/j.issn.1000-1646.2021.03.13,NaN,JOUR,Scopus,1,6,In view of the problems of low load rate of si...,2021.0
9190,Large-scale mesoscopic transport in nanostruct...,"[Hai-Jing, Zhang]",2013,Wuli,10.7693/wl201.30701,NaN,JOUR,DTU FindIt Eng,1,1,,2013.0
9191,Mechanistic pathways of mercury removal fromth...,"[Silva, P.J., Rodrigues, V.]",2015,PeerJ,10.7717/peerj.1127,NaN,JOUR,Scopus,1,5,Bacterial populations present in Hg-rich envir...,2015.0


In [9]:
# Example of using the title keyword filter
title_keywords_to_exclude = ['chemi','metal', 'cataly', ' ion', '-ion', 'virus','molec', 'bio', 'nano', 'crystal', 'atom', 'proton', 'protein', 'organic', \
                            'organom', 'photon', 'laser', 'transistor', 'josephson', 'superconduct', 'der Waals', 'quantum dot', 'graphen', 'graphit', 'cancer', 'silicon', \
                            'semiconduct', 'spectroscop', 'benzene', 'vinyl', 'ethyl', 'sulfuri', 'DNA', 'alcohol', 'polymer', 'fluor', 'chlorophyl', 'caroten', 'bacter', \
                            'microbi', 'mosfet', 'oxid', 'optic', 'chromat', 'dipole', 'Einstein', 'aromat', ' Si ', 'TiO2', 'InGaN', 'indium', 'AlxGa1', 'epitaxy',\
                            'acid', 'solvent', 'myosin', 'nucleotid', 'ribonucle', 'Cs2Te', ' opto', 'substrate', 'MOSHEMT', 'photodetect', 'nitrid', 'gallium','Ti3C2T','zinc',\
                            'ZnTeSe', 'quantum-dot', 'phospho', 'phenyl', 'Chlor', 'MoTe2', 'therapy', 'selenium', 'dosimetry', 'patient', 'sound', 'Fermi','Weyl', 'Dirac', \
                            'hydrogen', 'oxyge', 'phonon', 'hydrocarbo', 'lithium', 'neutron', 'supernov', 'sodium', 'majorana', 'Bogoliubov', 'ferromagn','cardio', ' gluon ', \
                            'neutrino', 'dark matter', 'meson', 'parton','quark', 'baryon', 'InGaAs', 'Rb2Sn', 'InAsSb','thiophene','perovskite', 'copper', 'sulfide', 'tungsten',\
                            'GaN-', 'AlGaN', 'Cathodo', 'SrTiO3', 'LaAlO3', 'photosynth', 'electrode', 'planet', 'pharma', 'medical imag', 'exciton', 'cosmo', 'ethanol', 'drug',\
                            'metaboli', 'chiral', 'CuO/ZnO', 'naphthal', 'pentadi', 'black hole', '/GaN', 'GaN/'] #Do not exclude BOSON

# Assuming is your dataframe with articles
filtered_df = filter_out_by_keywords(filtered_df, title_keywords_to_exclude, 'Title', verbose=True)
filtered_df

Filtered out 5093 articles (55.4%) containing these keywords in titles: chemi, metal, cataly,  ion, -ion, virus, molec, bio, nano, crystal, atom, proton, protein, organic, organom, photon, laser, transistor, josephson, superconduct, der Waals, quantum dot, graphen, graphit, cancer, silicon, semiconduct, spectroscop, benzene, vinyl, ethyl, sulfuri, DNA, alcohol, polymer, fluor, chlorophyl, caroten, bacter, microbi, mosfet, oxid, optic, chromat, dipole, Einstein, aromat,  Si , TiO2, InGaN, indium, AlxGa1, epitaxy, acid, solvent, myosin, nucleotid, ribonucle, Cs2Te,  opto, substrate, MOSHEMT, photodetect, nitrid, gallium, Ti3C2T, zinc, ZnTeSe, quantum-dot, phospho, phenyl, Chlor, MoTe2, therapy, selenium, dosimetry, patient, sound, Fermi, Weyl, Dirac, hydrogen, oxyge, phonon, hydrocarbo, lithium, neutron, supernov, sodium, majorana, Bogoliubov, ferromagn, cardio,  gluon , neutrino, dark matter, meson, parton, quark, baryon, InGaAs, Rb2Sn, InAsSb, thiophene, perovskite, copper, sulfide, tu

,Title,Authors,Year,Journal,DOI,Publisher,ReferenceType,Database,Q,T,Abstract,year
0,10th EAI International Conference on Industria...,NaN,2024,Lecture Notes of the Institute for Computer Sc...,NaN,NaN,JOUR,Scopus,7,7,The proceedings contain 18 papers. The special...,2024.0
1,10th International Conference on Emerging Ubiq...,NaN,2019,Procedia Computer Science,NaN,NaN,CONF,Scopus,7,6,The proceedings contain 126 papers. The topics...,2019.0
2,10th International Conference on Monte Carlo a...,NaN,2013,Springer Proceedings in Mathematics and Statis...,NaN,NaN,CONF,Scopus,1,6,The proceedings contain 34 papers. The special...,2013.0
3,10th International Symposium on Intelligence C...,NaN,2019,Communications in Computer and Information Sci...,NaN,NaN,JOUR,Scopus,1,6,The proceedings contain 32 papers. The special...,2019.0
4,11th Conference on Advanced Computer Architect...,NaN,2016,Communications in Computer and Information Sci...,NaN,NaN,JOUR,Scopus,1,5,The proceedings contain 17 papers. The special...,2016.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4095,First-principles based ballistic transport sim...,"[Chang, P., Liu, X., Liu, F., Du, G.]",2019,Japanese Journal of Applied Physics,10.7567/1347-4065/aafb4f,NaN,JOUR,Scopus,1,5,Ballistic performance of monolayer and few-lay...,2019.0
4096,Routing protocol of fiber quantum communicatio...,"[Yuan, X.-H., Li, C.-W.]",2017,Kongzhi Lilun Yu Yingyong/Control Theory and A...,10.7641/CTA.2017.70652,NaN,JOUR,Scopus,1,6,A routing protocol is proposed for the fiber q...,2017.0
4097,Research on quantum cognition and decision mak...,"[Song, Q.-Y., Fu, W.-P., Wang, W., Gao, Z.-Q.,...",2022,Kongzhi Lilun Yu Yingyong/Control Theory and A...,10.7641/CTA.2021.00892,NaN,JOUR,Scopus,1,5,Autonomous vehicles will share roads with huma...,2022.0
4098,A clutter suppression method utilizing time de...,"[Zhai, Y., Wu, J., Wang, D.]",2015,Hsi-An Chiao Tung Ta Hsueh/Journal of Xi'an Ji...,10.7652/xjtuxb201512008,NaN,JOUR,Scopus,1,6,A novel clutter suppression method utilizing w...,2015.0


In [10]:
#Optional: Export to Excel
#output_file = 'ArticlesToAnalize.xlsx'
#filtered_df.to_excel(output_file, index=False)

#print(f"DataFrame successfully written to {output_file}")